In [1]:
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr
!pip install facexlib
!pip install gfpgan
!pip install -r requirements.txt
!apt-get install -y imagemagick
!pip install wand
!pip install -q gradio

Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 36.29 MiB/s, done.
Resolving deltas: 100% (407/407), done.
/content/Real-ESRGAN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.

In [19]:
# 🌐 Web Interface with Upload → Enhance → Display (ngrok)
from PIL import Image
import os, cv2, numpy as np
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage import img_as_ubyte
from wand.image import Image as WandImage
import gradio as gr
os.environ['MAGICK_HOME'] = '/usr'
!python setup.py develop

os.makedirs('inputs', exist_ok=True)

def enhance_brightness(img_path):
    import cv2
    img = cv2.imread(img_path)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # Step 2: Histogram equalization on V channel
    v_eq = cv2.equalizeHist(v)
    hsv_eq = cv2.merge((h, s, v_eq))
    enhanced = cv2.cvtColor(hsv_eq, cv2.COLOR_HSV2BGR)

    cv2.imwrite("/content/inputs/output_brightness.jpg", enhanced)

def process_image(image):
    os.makedirs("/content/inputs", exist_ok=True)
    img_path = "/content/inputs/inputimg.jpg"
    image.save(img_path)
    with WandImage(filename="/content/inputs/inputimg.jpg") as img:
      img.sharpen(radius=8, sigma=4)
      img.save(filename='/content/inputs/output_sharp.jpg')
    enhance_brightness('/content/inputs/output_sharp.jpg')

    img = cv2.imread("/content/inputs/output_brightness.jpg")
#bilateral = cv2.bilateralFilter(img, d=75, sigmaColor=32, sigmaSpace=32)
#cv2.imwrite("output_bilateral.jpg", bilateral)

    nlm = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
    cv2.imwrite("/content/inputs/output_nlm.jpg", nlm)

    img = cv2.imread("/content/inputs/output_nlm.jpg")
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    clahe = cv2.createCLAHE(clipLimit=0.8, tileGridSize=(4, 4))
    v = clahe.apply(v)
    merged = cv2.merge((h, s, v))
    enhanced = cv2.cvtColor(merged, cv2.COLOR_HSV2BGR)
    cv2.imwrite("/content/inputs/output_contrast.jpg", enhanced)

    # 5. Super Resolution using Real-ESRGAN (EDSR)
    !python inference_realesrgan.py -n RealESRGAN_x4plus -i /content/inputs/output_contrast.jpg -dn 1 --outscale 2.5  --tile 800
    img = Image.open('results/output_contrast_out.jpg')
    return img

gr.Interface(
    fn=process_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Image(label="Enhanced (Contrast)"),
    title="AI Image Enhancer for Real Estate",
    description="Upload a real estate photo. AI will enhance contrast, brightness, sharpness, and remove noise."
).launch(share=True, debug=True)

/usr/local/lib/python3.11/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.11/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        *****

Testing 0 output_contrast
	Tile 1/24
	Tile 2/24
	Tile 3/24
	Tile 4/24
	Tile 5/24
	Tile 6/24
	Tile 7/24
	Tile 8/24
	Tile 9/24
	Tile 10/24
	Tile 11/24
	Tile 12/24
	Tile 13/24
	Tile 14/24
	Tile 15/24
	Tile 16/24
	Tile 17/24
	Tile 18/24
	Tile 19/24
	Tile 20/24
	Tile 21/24
	Tile 22/24
	Tile 23/24
	Tile 24/24
Testing 0 output_contrast
	Tile 1/24
	Tile 2/24
	Tile 3/24
	Tile 4/24
	Tile 5/24
	Tile 6/24
	Tile 7/24
	Tile 8/24
	Tile 9/24
	Tile 10/24
	Tile 11/24
	Tile 12/24
	Tile 13/24
	Tile 14/24
	Tile 15/24
	Tile 16/24
	Tile 17/24
	Tile 18/24
	Tile 19/24
	Tile 20/24
	Tile 21/24
	Tile 22/24
	Tile 23/24
	Tile 24/24
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b57d5d7d759c43c14f.gradio.live
